# Explore to clean columns

In [0]:
df = spark.sql("SELECT * FROM marathos_cat.bronze.raw_marathos")
df.limit(5).display()

## Front and empty spaces

In [0]:
#CHECK IF CAN BE DELETED 
# from pyspark.sql.functions import trim, col

#df = df_cleaned

#for column, dtype in df_cleaned.dtypes:
#    if dtype == 'string':
#        df_cleaned = df_cleaned.withColumn(column, trim(col(column)))

#(df_cleaned.select("athlete_club")
#    .distinct()
#    .limit(20)
#    .display()
#)

## All in snake_case

In [0]:
# --- checking for utils only ---
import sys
sys.path.append("/Workspace/Users/giraudcgc@gmail.com/lab1_big_data_cloud_christophe_giraud/marathos_christophe_giraud/utils")

from utils import rename_columns_to_snake_case, to_snake_case

# --- Sanity check ---
print(to_snake_case("Event Distance/Length"))  # expected: event_distance_or_length

# --- Assignign df_cleaned ---
df_cleaned = rename_columns_to_snake_case(df)

In [0]:
import os
print(os.getcwd())

In [0]:
# --- Trimming first ---
from pyspark.sql.functions import trim, col


for column, dtype in df_cleaned.dtypes:
    if dtype == 'string':
        df_cleaned = df_cleaned.withColumn(column, trim(col(column)))

## Year of event

From online research "The first competitive marathon was held as the final event of the track and field program at the 1896 Summer Olympics in Athens, Greece, on April 10, 1896". I will consider it.

In [0]:
# --- summary ---
print("===== Summary =====")
df_cleaned.select("year_of_event").summary().show()

# --- mini and max considered valid dates ---
print(f"Before 1896: {df_cleaned.filter(col('year_of_event') < 1896).count()}")
print(f"After 2022: {df_cleaned.filter(col('year_of_event') > 2022).count()}")

# --- removing impossible dates from the dataset ---
df_cleaned = df_cleaned.filter(
    (col('year_of_event') >= 1896) & (col('year_of_event') <= 2022)
)

## Converting column "event_dates" from str to date

In [0]:
from pyspark.sql.functions import col, to_date

# --- to_date ---
from pyspark.sql.functions import try_to_date

df_cleaned = df_cleaned.withColumn("event_dates", try_to_date(col("event_dates"), "dd.MM.yyyy"))
# testing it
df_cleaned.printSchema()

## Event name

In [0]:
from pyspark.sql.functions import col, regexp_replace

# --- Quick view ---
print(f"====== Checking the titles =====")
(df_cleaned.filter(col("event_name").rlike(r'[^a-zA-Z0-9\s\(\)]')) 
    .select("event_name") 
    .distinct() 
    .limit(5000) 
    .display()
)

'''I will keep the titles rich with details or some symbols which are meaningfull and would make the dashboard readable. After testing, some titles were so "trimmed or normalized" that it became impossible to understand or read the event name'''

# --- Removing some characters like: "", <, > ---
df_cleaned = df_cleaned.withColumn(
    "event_name",
    regexp_replace(col("event_name"), r'[\"<>]', '')
)

# --- Checking ---
print(f"Remaining quotes: {df_cleaned.filter(col('event_name').contains('\"')).count()}")
print(f"Remaining <: {df_cleaned.filter(col('event_name').contains('<')).count()}")
print(f"Remaining >: {df_cleaned.filter(col('event_name').contains('>')).count()}")

## Column event_distance_or_length

In [0]:
'''Here I am first removing the elements discussed in the classroom like values containing "d" for "days", or "Etappen".'''

# --- Counting values containing "d" for "days" ---
# Used LLM for the regex part
d_count = df_cleaned.filter(col("event_distance_or_length").rlike(r"^\d+\.?\d*d")).count()
#print(d_count)

# --- Then counting values containing "Etappen" --- 
etappen_count = df_cleaned.filter(col("event_distance_or_length").contains("Etappen")).count()
#print(etappen_count)

total = df_cleaned.count()
#print(total)

print(f"Total rows: {total}")
print(f"Rows with d: {d_count}")
print(f"Rows with Etappen: {etappen_count}")
print(f"Rows remaining after removal: {total - d_count - etappen_count}")

# -- Now actually removing them by "ignoring" them with " ~ " ---
df_cleaned = df_cleaned.filter(~col("event_distance_or_length").rlike(r"^\d+\.?\d*d"))
df_cleaned = df_cleaned.filter(~col("event_distance_or_length").contains("Etappen"))


In [0]:
'''Here I separate the numbers from the letters or units of measurement like km, miles etc.. to supress or rename and regroup them eventually'''

from pyspark.sql.functions import regexp_extract

df_cleaned = df_cleaned.withColumn(
    "unit_measure",
    regexp_extract(col("event_distance_or_length"), r"[a-zA-Z]+", 0)
)

df_cleaned.groupBy("unit_measure").count().orderBy("count", ascending=False).show()

In [0]:
# --- Suppressing the uncertain ones ---
df_cleaned = df_cleaned.filter(~col("unit_measure").isin(["None", "m", "x", ""]))

# Checking
df_cleaned.count()

In [0]:
# --- Standardization of the relevant ones ---
from pyspark.sql.functions import when

df_cleaned = df_cleaned.withColumn(
    "unit_measure",
    when(col("unit_measure").isin(["km", "Km", "k", "K"]), "km")
    .when(col("unit_measure").isin(["mi", "Miles", "miles", "Mile", "mile"]), "mi")
    .when(col("unit_measure") == "h", "h")
    .otherwise(col("unit_measure"))
)

# added later once working on the last silver question
df_cleaned = df_cleaned.withColumn(
    "event_type",
    when(col("unit_measure") == "km", "distance")
    .when(col("unit_measure") == "h", "length")
    .otherwise(None)
)

# --- checking ---
print(f"===== Checking the standardisation =====")
df_cleaned.groupBy("unit_measure").count().orderBy("count", ascending=False).show()

# --- Checking ---
print(f"===== Confirmation =====")
df_cleaned.limit(30).display()


In [0]:
'''Above I have extracetd the unit measure, here I will extract the numric value and convert "miles" to "km" '''

# --- Extraction of the numeric part --- 
from pyspark.sql.functions import round as spark_round

# Below Regex: r"(\d+\.?\d*)", 1) was produced by LLM
df_cleaned = df_cleaned.withColumn(
    "unit_value", 
    spark_round(
        when(col("unit_measure") == "mi", regexp_extract(col("event_distance_or_length"), r"(\d+\.?\d*)", 1)
        .cast("double") * 1.60934)
        .otherwise(regexp_extract(col("event_distance_or_length"), r"(\d+\.?\d*)", 1).cast("double")),
        2
    )
).withColumn(
    "unit_measure",
    when(col("unit_measure") == "mi", "km").otherwise(col("unit_measure"))
)

df_cleaned.select("unit_value", "unit_measure").show(10)

In [0]:
# --- Need to check df_cleaned so far ---
df_cleaned.limit(200).display()

# Athlete_performance

In [0]:
# starting it again

# --- df_cleaned so far ---
print(f"===== Starting with quick overview =====")
df_cleaned.limit(1).display()


# --- Performance_unit ---
print(f"===== Extracting perf_unit =====")
df_cleaned = df_cleaned.withColumn(
    "performance_unit",
    regexp_extract(col("athlete_performance"), r"[a-zA-Z]+", 0)
)
df_cleaned.limit(1).display()

# --- Convert athlete_performance to numeric ---
print(f"===== Adding athlete_performance_value =====")
df_cleaned = df_cleaned.withColumn(
    "athlete_performance_value",
    when(
        col("performance_unit") == "h",
        # convert HH:MM:SS to decimal hours
        regexp_extract(col("athlete_performance"), r"(\d+):(\d+):(\d+)", 1).cast("double") +
        regexp_extract(col("athlete_performance"), r"(\d+):(\d+):(\d+)", 2).cast("double") / 60 +
        regexp_extract(col("athlete_performance"), r"(\d+):(\d+):(\d+)", 3).cast("double") / 3600
    ).otherwise(
        # extract numeric value for km
        regexp_extract(col("athlete_performance"), r"(\d+\.?\d*)", 1).cast("double")
    )
)

# --- rounding ---
df_cleaned = df_cleaned.withColumn(
    "athlete_performance_value",
    spark_round(col("athlete_performance_value"), 2)
)
df_cleaned.limit(1).display()

# --- Checking ----
print(f"===== Checking the relevant columns =====")
df_cleaned.select("event_distance_or_length","event_type","athlete_performance","unit_value","unit_measure", "athlete_performance_value", "performance_unit").show(1)


## Average speed

In [0]:
df_cleaned.select("athlete_average_speed").summary().show()

In [0]:
'''As per online research, the "absolute maximum average speed for a marathon runner is approximately 21.1 km/h or  13.1mph which corresponds to the official marathon world record of 2h 35s. I will exclude all fastest results'''

from pyspark.sql.functions import round as spark_round, col, when

# --- Viewing the average speeds ---
print(f"===== Column showing the average speeeds ======")
(df_cleaned.select("athlete_average_speed")
    .distinct()
    .orderBy("athlete_average_speed", ascending=False)
    .display()
)
# --- datatype changed to 'double'? ---
print(f"===== Datatype =====") 
df_cleaned.select("athlete_average_speed").printSchema()

# Conversion to float would work but double is the PySpark default and rounding to 1 decimal ---
# SHOULD I SHOW IT?
df_cleaned = df_cleaned.withColumn(
    "athlete_average_speed",
    spark_round(
        when(col("athlete_average_speed").rlike(r"^\d+\.?\d*$"), 
             col("athlete_average_speed").cast("double"))
        .otherwise(None),
        1
    )
)

# --— filtering ---
df_cleaned = df_cleaned.filter(
    (col("athlete_average_speed") > 0) & (col("athlete_average_speed") <= 22)
)



In [0]:
df_cleaned.select("athlete_average_speed").summary().show()

Note: Reasonable to say that minimun average speed shoud be around 5km/h
+ if time, will add a graph + maybe exclude all below 5km

## Athlete club

sample after clenaing is still bad!!!

In [0]:
'''Many runners don't belong to a club in reality. Will move fast, so to speak, to the next column'''
from pyspark.sql.functions import trim, regexp_replace

# --- Quick look --- 
print("===== Look at a sample =====")
(df_cleaned.select("athlete_club")
    .distinct()
    .limit(5)
    .display()
)

# --- How many clubs? ---
print(f"===== Number of clubs =====")
print(f"Distinct clubs: {df_cleaned.select('athlete_club').distinct().count()}")

# --- datatype ---
print(f"===== Data type =====")
df_cleaned.select("athlete_club").printSchema()

# --- Removing anything which is not a letter or number ---
df_cleaned = df_cleaned.withColumn(
    "athlete_club",
    trim(regexp_replace(col("athlete_club"), r'^[^a-zA-Z0-9\u4e00-\u9fff]+|[^a-zA-Z0-9\u4e00-\u9fff]+$', ''))
)

# --- Set empty values to null ---
df_cleaned = df_cleaned.withColumn(
    "athlete_club",
    when(col("athlete_club") == '', None)
    .otherwise(col("athlete_club"))
)

# --- Sample after cleaning ---
print("===== Sample after cleaning =====")
(df_cleaned.select("athlete_club")
    .filter(col("athlete_club").isNotNull())
    .distinct()
    .orderBy("athlete_club")
    .limit(100)
    .display()
)


## Athlete country


In [0]:
from pyspark.sql.functions import upper

# --- Quick view  ---
print(f"===== Descending country view =====")
(df_cleaned.select("athlete_country")
    .orderBy("athlete_country", ascending=False)
    .display())

# --- nulls ---
print(f"===== How many nulls? =====")
print(f"Nulls: {df_cleaned.filter(col('athlete_country').isNull()).count()}")

# --- Unknown values ---
print("===== Non assigned value 'XXX' =====")
print(f"XXX: {df_cleaned.filter(col('athlete_country') == 'XXX').count()}")

# --- Not in capital letters ---
print(f"===== In lowercase =====") 
print(f"{df_cleaned.filter(~col('athlete_country').rlike(r'^[A-Z]{3}$')).count()}")

# --- Sample of non-standard values ---
print("===== Non standard =====")
(df_cleaned.filter(~col('athlete_country').rlike(r'^[A-Z]{3}$'))
    .select("athlete_country")
    .distinct()
    .display())

# --- FIxing the issue ---
# --- fix ---
df_cleaned = df_cleaned.withColumn("athlete_country", upper(col("athlete_country")))

In [0]:
df_cleaned.select("athlete_country").distinct().orderBy("athlete_country").display()

## Athlete year of birth

In [0]:
# --- Checking datatype again ---
print(f"===== Datatype =====")
df_cleaned.select("athlete_year_of_birth").printSchema()

# --- Changing to integer ---
print(F"===== Datatype changed to integer =====")
df_cleaned = df_cleaned.withColumn(
    "athlete_year_of_birth",
    col("athlete_year_of_birth").cast("int")
)
df_cleaned.select("athlete_year_of_birth").printSchema()

# --- Quick overview 
print(f"===== Statistics =====")
df_cleaned.select("athlete_year_of_birth").summary().show()

(df_cleaned.select("year_of_event")
    .distinct()
    .orderBy("year_of_event")
    .display()
)

'''
Cleaning considerations. Marathons are dated from 1996 to 20222 in the dataset. 
In general, one must be 18 years old to compete. There are exceptionnel people aged 100+ who have run a marathon. 
So: 
- the latest possible DOB is: 2022 - 18 = 2004 (minimum 18 years old runner)
- 'the earliest/oldest "reasonable": 1922 (100 years old in 2022, generous limit)
'''

# --- To be removed --- TO REDO THIS !!!!!!
print(f"===== Removing outliers counts =====")
print(f"Born before 1922: {df_cleaned.filter(col('athlete_year_of_birth') < 1922).count()}")
print(f"Born after 2004: {df_cleaned.filter(col('athlete_year_of_birth') > 2004).count()}")

# --- Creating a new column to know the runner's age ---
df_cleaned = df_cleaned.withColumn(
    "age_at_event",
    col("year_of_event") - col("athlete_year_of_birth")
)

# --- Checking --- 
print(f"===== Sample to confirm ======")
df_cleaned.select("year_of_event", "athlete_year_of_birth", "age_at_event").show(10)
# --- checking the schema ----
df_cleaned.printSchema()

# --- Nulls ---
print("===== Case of the nulls =====")
print(f"Nulls in athlete_year_of_birth: {df_cleaned.filter(col('athlete_year_of_birth').isNull()).count()}")
print(f"Here I keep the nulls because we can always assume that they were at least 18 years old and it is 8% of the total rows ")

## Event number of finishers

In [0]:
# --- Quick overview ---
print(f"===== statistics =====")
df_cleaned.select("event_number_of_finishers").summary().show()

In [0]:
''' I will consider here that the only anomaly is "min 0" as it is impossible that nobody finishes a marathon '''

print(f"===== Events with zero finisher =====")
print(df_cleaned.filter(col('event_number_of_finishers') == 0).count())

'''Extra note: I will remove the rows here because, contrary to the athlete_club column were "null" means that runners have not club, theses nulls are suspect'''

# --- Suppressig them from df_clean ---
df_cleaned = df_cleaned.filter(col("event_number_of_finishers") > 0)

# --- Checking ---
print(f"=== Remaining events with zero finisher ===")
print(df_cleaned.filter(col('event_number_of_finishers') == 0).count())
      

## Athlete gender

In [0]:
# --- datatype ---
print(f"===== Datatype =====")
df_cleaned.select("athlete_gender").printSchema()

# --- Quick overview ---
print("===== Summary =====")
df_cleaned.select("athlete_gender").summary().show()

# --- distinct values ---
print("===== Distinct values =====")
(df_cleaned.select("athlete_gender")
    .distinct()
    .display()
)

# --- Count per gender ---
print("===== Count per value =====")
df_cleaned.groupBy("athlete_gender").count().orderBy("count", ascending=False).show()

# --- The nulls ---
print(f"===== Nulls =====")
print(f"{df_cleaned.filter(col('athlete_gender').isNull()).count()}")

# --- Removing the non F or M ---
df_cleaned = df_cleaned.filter(col("athlete_gender").isin(["M", "F"]))

# --- Checking ---
print(f"===== Remainig genders =====")
df_cleaned.groupBy("athlete_gender").count().show()


## Athlete age category

Note: The column athele_age_category contains letters "M" for "Men" and "W" for "Women" while the column "athlete_gender" contains "M" for "Male" and "F" for "Female". <br>
To avoid confusion and questions in a work environement, I will set all the "W" to "F" in athlete_age_category

In [0]:
# --- datatype ---
print(f"===== Datatype =====")
df_cleaned.select("athlete_age_category").printSchema()

# --- count ---
print("===== Count per value =====")
df_cleaned.groupBy("athlete_age_category").count().orderBy("count", ascending=False).display()

# --- nulls ---
print(f"Nulls: {df_cleaned.filter(col('athlete_age_category').isNull()).count()}")

In [0]:
'''We can see 1 "F35  and 493598 nulls'''

# --- For F35, checking the gender in the column athlete_gender ---
print(f"===== F35 gender =====")
df_cleaned.filter(col("athlete_age_category") == "F35").select("athlete_age_category", "athlete_gender").show(10)

# --- Converting all W to F ---
df_cleaned = df_cleaned.withColumn(
    "athlete_age_category",
    when(col("athlete_age_category").startswith("W"),
         regexp_replace(col("athlete_age_category"), "^W", "F"))
    .otherwise(col("athlete_age_category"))
)

# --- Checking ---
print(f"===== Confirming the change and distinct values =====")
df_cleaned.groupBy("athlete_age_category").count().orderBy("athlete_age_category").display()

Note 2: Many nulls = 493598 and there is no logical way to deduce the right value. We could confirm the gender with column athlete_gender but not the age category. TO CHECK LATER !!!!

## Athlete ID

In [0]:
# --- datatype ---
print(f"===== Datatype =====")
df_cleaned.select("athlete_id").printSchema()

# --- nulls ---
print(f"===== Nulls =====")
print(df_cleaned.filter(col('athlete_id').isNull()).count())

# --- rows & athlete id ---
print(f"===== Number of rows Vs athlete id =====")
print(f"Total rows: {df_cleaned.count()}")
print(f"Distinct athlete_ids: {df_cleaned.select('athlete_id').distinct().count()}")

'''There are many more events than athelte_id, so checking for duplicates or if an athlete runs different marathons with the same id? 
A row is a duplicate if ALL the below columns are identical:athlete_id,event_name,year_of_event,athlete_performance,unit_value         
'''

# --- check for duplicates ---
print(f"===== Duplicates =====")
duplicates = df_cleaned.groupBy(
    "athlete_id",
    "event_dates",
    "event_name",
    "year_of_event",
    "athlete_performance",
    "unit_value",
    "athlete_gender",
    "athlete_age_category"
).count().filter(col("count") > 1)

print(f"duplicates: {duplicates.count()}")
duplicates.orderBy("athlete_id").display()

In [0]:

# test again
print(f"===== checking id 105 =====")
df_cleaned.filter(
    (col("athlete_id") == 105) &
    (col("event_name") == "Across the Years, 48h (USA)") &
    (col("year_of_event") == 2019)
).display()

Note: The above shows that there are identical but for the athelete_club column.  

In [0]:
# Count total rows vs. unique athlete IDs
#total_rows = df.count()
total_rows = df_cleaned.count()
unique_athletes = df_cleaned.select("athlete_id").distinct().count()

print(f"Total Records: {total_rows}")
print(f"Unique Athletes: {unique_athletes}")

# Look at an athlete who appears multiple times to inspect the data
from pyspark.sql import functions as F

duplicate_athletes = df_cleaned.groupBy("athlete_id").count().filter("count > 1")
duplicate_athletes.show(5)

In [0]:
# --- Deduplication ---
df_cleaned = df_cleaned.sort(
    col("athlete_id"),
    col("event_name"),
    col("event_dates"),
    col("athlete_club").desc(),
    col("athlete_year_of_birth").desc(),
    col("athlete_age_category").desc()
)

df_cleaned = df_cleaned.dropDuplicates([
    "athlete_id",
    "event_name",
    "event_dates",
    "event_distance_or_length"
])

print(f"===== Rows after deduplication =====")
print(df_cleaned.count())

In [0]:
# --- Need to see the schema before dimensional model---
print(f"===== Schema overview =====")
df_cleaned.printSchema()


In [0]:
%sql
SELECT * FROM marathos_cat.gold.dim_date;